# Task 5: Strategy Backtesting

Compare optimized portfolio vs 60% SPY / 40% BND benchmark.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from src.backtesting import compare_strategies
from src.data_loader import combine_close_prices, fetch_all_assets
from src.portfolio import build_expected_returns_vector, compute_covariance_matrix, optimize_portfolios
from src.forecasting import fit_auto_arima, forecast_arima
from src.preprocessing import clean_asset_frame
from src.portfolio import annualized_forecast_return_from_prices


In [ ]:
asset_data = {t: clean_asset_frame(df) for t, df in fetch_all_assets().items()}
prices = combine_close_prices(asset_data)
returns = prices.pct_change().dropna()

train_returns = returns.loc[:'2024-12-31']
backtest_returns = returns.loc['2025-01-01':'2026-01-31']

tsla_train = prices.loc[:'2024-12-31', 'TSLA']
arima_model = fit_auto_arima(tsla_train, seasonal=True, m=5)
future_prices = forecast_arima(arima_model, 252)
tsla_expected = annualized_forecast_return_from_prices(
    float(tsla_train.iloc[-1]), future_prices, 252
)

expected_returns = build_expected_returns_vector(train_returns, tsla_expected)
cov_matrix = compute_covariance_matrix(train_returns)
strategy_weights = optimize_portfolios(expected_returns, cov_matrix)['max_sharpe']['weights']
benchmark_weights = {'TSLA': 0.0, 'SPY': 0.6, 'BND': 0.4}

metrics, strat_cum, bench_cum = compare_strategies(
    backtest_returns, strategy_weights, benchmark_weights
)
metrics

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(strat_cum.index, strat_cum.values, label='Optimized Strategy')
plt.plot(bench_cum.index, bench_cum.values, label='60/40 SPY-BND Benchmark')
plt.title('Cumulative Returns: Strategy vs Benchmark')
plt.ylabel('Growth of $1')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Conclusion

This backtest compares a forecast-informed optimal portfolio against a passive 60/40 benchmark over Jan 2025 – Jan 2026. Outperformance would suggest the model-driven allocation adds value; underperformance highlights EMH constraints and the limits of a single-asset forecast view.

**Limitations:** Static weights, no transaction costs, short window, and look-ahead bias in covariance estimation if not carefully separated.